In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv("../data/raw/sram_fault_dataset.csv")
print("Dataset shape:", df.shape)

Dataset shape: (450, 33)


In [4]:
# 1. Check whether SNM drop is a derived feature
df["snm_drop_check"] = (
    df["snm_hold_v"] - df["snm_read_v"]
)

print("SNM drop verification:")
print(
    np.allclose(
        df["snm_drop_v"],
        df["snm_drop_check"],
        atol=1e-6
    )
)

# 2. Check constant columns
print("\nConstant columns:")
print(df.nunique()[df.nunique() == 1])

# 3. Inspect capped delay values
print("\nWrite delay capped samples:")
print(df[df["t_write_ps"] >= 20000][
    ["fault_class", "t_write_ps", "converged"]
])

print("\nSense delay capped samples:")
print(df[df["t_sense_ps"] >= 20000][
    ["fault_class", "t_sense_ps", "converged"]
])

# 4. Inspect extreme SNM ratios
print("\nLargest SNM ratios:")
print(
    df.nlargest(10, "snm_ratio")[
        ["snm_hold_v", "snm_read_v", "snm_ratio", "fault_class"]
    ]
)

SNM drop verification:
True

Constant columns:
snm_asym_hold_v    1
dtype: int64

Write delay capped samples:
     fault_class  t_write_ps  converged
13             2     20000.0          0
14             2     20000.0          0
15             2     20000.0          0
16             2     20000.0          0
17             2     20000.0          0
..           ...         ...        ...
433            2     20000.0          0
434            2     20000.0          0
435            2     20000.0          0
436            2     20000.0          0
437            2     20000.0          0

[75 rows x 3 columns]

Sense delay capped samples:
     fault_class  t_sense_ps  converged
13             2     20000.0          0
14             2     20000.0          0
15             2     20000.0          0
16             2     20000.0          0
17             2     20000.0          0
..           ...         ...        ...
433            2     20000.0          0
434            2     20000.0          

In [5]:
# Target variable
target = "fault_class"

# Columns that identify samples or simulation conditions
identifier_columns = [
    "sample_id",
    "corner_id"
]

# Categorical features
categorical_features = [
    "pvt_type",
    "fault_name"
]

# Potential numerical features
numerical_features = df.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Target:", target)

print("\nIdentifier columns:")
print(identifier_columns)

print("\nCategorical columns:")
print(categorical_features)

print("\nNumerical columns:")
print(numerical_features)

Target: fault_class

Identifier columns:
['sample_id', 'corner_id']

Categorical columns:
['pvt_type', 'fault_name']

Numerical columns:
['vdd_v', 'temperature_c', 'fault_class', 'severity_value', 'i_ddq_uA', 't_write_ps', 'i_write_peak_mA', 'dv_bl_strobe_mV', 't_sense_ps', 'v_bump_mV', 'hold_pass', 'write_pass', 'read_stable_pass', 'snm_hold_v', 'snm_read_v', 'snm_asym_hold_v', 'snm_asym_read_v', 'snm_drop_v', 'is_bistable', 'iddq_norm', 't_write_norm', 'dv_bl_norm', 'snm_hold_norm', 'snm_read_norm', 'read_write_delay_ratio', 'snm_ratio', 'dv_bl_vdd_ratio', 'converged', 'snm_drop_check']


In [6]:
print("Target distribution:")
print(df["fault_class"].value_counts().sort_index())

print("\nTarget distribution percentage:")
print(
    df["fault_class"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

Target distribution:
fault_class
0     15
1    180
2     75
3     90
4     90
Name: count, dtype: int64

Target distribution percentage:
fault_class
0     3.333333
1    40.000000
2    16.666667
3    20.000000
4    20.000000
Name: proportion, dtype: float64


In [7]:
# Compare numerical feature averages across fault classes
numeric_df = df.select_dtypes(include=["int64", "float64"])

feature_comparison = numeric_df.groupby(
    df["fault_class"]
).mean()

print(feature_comparison.T)

fault_class                      0            1             2           3  \
vdd_v                     1.000000     1.000000      1.000000    1.000000   
temperature_c            37.400000    37.400000     37.400000   37.400000   
fault_class               0.000000     1.000000      2.000000    3.000000   
severity_value            0.000000  3116.666667   3700.000000   17.500000   
i_ddq_uA                  0.499240     0.290530      8.428619    0.133683   
t_write_ps               61.633333    73.979444  20000.000000   60.855556   
i_write_peak_mA           0.034733     0.034694      0.035013    0.029633   
dv_bl_strobe_mV         235.406667   228.749444      0.000000  218.176667   
t_sense_ps               61.120000    61.464444  20000.000000   63.810000   
v_bump_mV               128.160000   128.143333    500.997333  142.061111   
hold_pass                 0.933333     0.988889      0.000000    0.988889   
write_pass                0.933333     0.988889      0.000000    0.955556   

In [8]:
# Investigate suspicious columns

print("FAULT TARGET VALUES")
print(df["fault_target"].value_counts(dropna=False))

print("\nFAULT TARGET vs FAULT CLASS")
print(pd.crosstab(df["fault_class"], df["fault_target"]))

print("\nSEVERITY VALUE BY FAULT CLASS")
print(
    df.groupby("fault_class")["severity_value"]
    .agg(["min", "max", "mean", "std"])
)

print("\nCONVERGENCE BY FAULT CLASS")
print(
    pd.crosstab(
        df["fault_class"],
        df["converged"],
        normalize="index"
    ) * 100
)

print("\nUNIQUE VALUES IN SUSPICIOUS FEATURES")
for col in [
    "fault_target",
    "severity_value",
    "converged",
    "snm_asym_hold_v"
]:
    print(f"{col}: {df[col].nunique()} unique values")

FAULT TARGET VALUES
fault_target
M5              90
M6              90
access_pair     90
storage_pair    90
Q_Qb            75
none            15
Name: count, dtype: int64

FAULT TARGET vs FAULT CLASS
fault_target  M5  M6  Q_Qb  access_pair  none  storage_pair
fault_class                                                
0              0   0     0            0    15             0
1             90  90     0            0     0             0
2              0   0    75            0     0             0
3              0   0     0            0     0            90
4              0   0     0           90     0             0

SEVERITY VALUE BY FAULT CLASS
               min      max         mean          std
fault_class                                          
0              0.0      0.0     0.000000     0.000000
1            200.0  10000.0  3116.666667  3473.965422
2            500.0  10000.0  3700.000000  3539.354041
3              5.0     30.0    17.500000     8.586964
4              5.0     

In [10]:
df.drop(columns=["snm_drop_check"], inplace=True, errors="ignore")